In [1]:
import kglab
import pickle
import rdflib 
import re
import torch
import json 

import networkx as nx
import pandas as pd
import numpy as np

from tqdm.notebook import tqdm
from collections import Counter
from sentence_transformers import SentenceTransformer, util
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
model = "Qwen"
prompt = "unstructured"
add_isco = False

In [3]:
df = pd.read_excel(f"../outputs/raw_outputs/generated_triples_coalesced.xlsx")

In [4]:
df.head()

,Unnamed: 0,model,prompt,id,text,triples
0,0,gemma,structured,"IT-administrator – få indflydelse på et setup,...",Vil du ind i en virksomhed i rivende udvikling...,"[('Job_Title', 'HAS_LOCATION', 'London, UK'), ..."
1,1,gemma,structured,"IT-administrator – få indflydelse på et setup,...","IT-administrator – få indflydelse på et setup,...","[('IT-administrator', 'OFFERS_POSITION', 'ALD ..."
2,2,gemma,structured,SQE Manager,For jobsøgere For arbejdsgivere Aqua d'Or Mine...,"[('Brande', 'HAS_LOCATION', 'Denmark'), ('Aqua..."
3,3,gemma,semi-structured,"IT-administrator – få indflydelse på et setup,...",Vil du ind i en virksomhed i rivende udvikling...,"[('Job title', 'requires_skill', 'IT operation..."
4,4,gemma,semi-structured,"IT-administrator – få indflydelse på et setup,...","IT-administrator – få indflydelse på et setup,...","[('Job', 'has_title', 'IT-administrator'), ('J..."


In [5]:
iscos = pd.read_csv("../pipeline/isco.csv", encoding="cp850")
iscos = iscos[iscos["ISCO_version"] == "ISCO-08"]

iscos.head()

,ISCO_version,major,major_label,sub_major,sub_major_label,minor,minor_label,unit,description
0,ISCO-08,1,Managers,11.0,"Chief Executives, Senior Officials and Legisla...",111,Legislators and Senior Officials,1111,Legislators
1,ISCO-08,1,Managers,11.0,"Chief Executives, Senior Officials and Legisla...",111,Legislators and Senior Officials,1112,Senior Government Officials
2,ISCO-08,1,Managers,11.0,"Chief Executives, Senior Officials and Legisla...",111,Legislators and Senior Officials,1113,Traditional Chiefs and Heads of Villages
3,ISCO-08,1,Managers,11.0,"Chief Executives, Senior Officials and Legisla...",111,Legislators and Senior Officials,1114,Senior Officials of Special-interest Organizat...
4,ISCO-08,1,Managers,11.0,"Chief Executives, Senior Officials and Legisla...",112,Managing Directors and Chief Executives,1120,Managing Directors and Chief Executives


In [6]:
namespaces = {
    "jip" : "http://company.com/property/",
    "jid" : "http://company.com/ontology/",
    "jie" : "http://company.com/entity/",
    "dbp": "http://dbpedia.org/property/",
    "dbo": "http://dbpedia.org/ontology/",
    "dbr": "http://dbpedia.org/resource/",
    "owl": "http://www.w3.org/2002/07/owl#",
    "rdfs": "http://www.w3.org/2000/01/rdf-schema#",
    "foaf": "http://xmlns.com/foaf/0.1/"
}

In [7]:
kg = kglab.KnowledgeGraph(
    name = "Jobindex KG",
    namespaces = namespaces,
    base_uri = "https://www.example.com/entity/"
)

In [8]:
# Define ontology lay-out

# Classes

# Candidate is a class, and a candidate is a person
kg.add(kg.get_ns("jid").Candidate, kg.get_ns("rdf").type, kg.get_ns("owl").Class)
kg.add(kg.get_ns("jid").Candidate, kg.get_ns("rdfs").subClassOf, kg.get_ns("foaf").Person)

# Function is a class, and is equivalent to employment
kg.add(kg.get_ns("jid").Position, kg.get_ns("rdf").type, kg.get_ns("owl").Class)

# Add educations and their ordering
edus = ["Doctorate", "PhD", "Masters_degree", "Bachelors_degree", "High_school_diploma"]

for i, edu in enumerate(edus):
    kg.add(eval(f"kg.get_ns('jie').{edu}"), kg.get_ns("rdf").type, kg.get_ns("jid").Education)
    
    for edu2 in edus[i+1:]:
        kg.add(eval(f"kg.get_ns('jie').{edu}"), kg.get_ns("jid").supersedes, eval(f"kg.get_ns('jie').{edu2}"))

# Company is a class, it offers functions, and candidates work there
kg.add(kg.get_ns("jid").Company, kg.get_ns("rdf").type, kg.get_ns("owl").Class)
kg.add(kg.get_ns("jid").Company, kg.get_ns("jip").offers_position, kg.get_ns("jid").Position)
kg.add(kg.get_ns("jid").Candidate, kg.get_ns("jip").has_worked_at, kg.get_ns("jid").Company)
kg.add(kg.get_ns("jid").Candidate, kg.get_ns("jip").has_worked_position, kg.get_ns("jid").Position)

### SINCE ISCO IS SO IMPORTANT, MAYBE TURN THE LLM PART INTO A PIPELINE --> EXTRACT TRIPLES, LINK TRIPLES TO ISCO
# An ISCO code is a class, and all sub-codes fall under that class
kg.add(kg.get_ns("jid").ISCO_code, kg.get_ns("rdf").type, kg.get_ns("owl").Class)
kg.add(kg.get_ns("jid").ISCO_unit, kg.get_ns("rdfs").subClassOf, kg.get_ns("jid").ISCO_code)
kg.add(kg.get_ns("jid").ISCO_minor, kg.get_ns("rdfs").subClassOf, kg.get_ns("jid").ISCO_code)
kg.add(kg.get_ns("jid").ISCO_sub_major, kg.get_ns("rdfs").subClassOf, kg.get_ns("jid").ISCO_code)
kg.add(kg.get_ns("jid").ISCO_major, kg.get_ns("rdfs").subClassOf, kg.get_ns("jid").ISCO_code)

# Add skill, language, and license class
kg.add(kg.get_ns("jid").Skill, kg.get_ns("rdf").type, kg.get_ns("owl").Class)
kg.add(kg.get_ns("jid").Language, kg.get_ns("rdf").type, kg.get_ns("owl").Class)
kg.add(kg.get_ns("jid").Certificate, kg.get_ns("rdf").type, kg.get_ns("owl").Class)

# Isco levels
for row in iscos.itertuples():
    # Unit
    kg.add(eval(f"kg.get_ns('jie').isco{row[8]}"), kg.get_ns("rdf").type, kg.get_ns("jid").ISCO_unit)
    kg.add(eval(f"kg.get_ns('jie').isco{row[8]}"), kg.get_ns("jip").falls_under, eval(f"kg.get_ns('jie').isco{row[6]}"))
    kg.add(eval(f"kg.get_ns('jie').isco{row[8]}"), kg.get_ns("rdfs").comment, rdflib.Literal(row[9]))
    
    # Minor
    kg.add(eval(f"kg.get_ns('jie').isco{row[6]}"), kg.get_ns("rdf").type, kg.get_ns("jid").ISCO_minor)
    kg.add(eval(f"kg.get_ns('jie').isco{row[6]}"), kg.get_ns("jip").falls_under, eval(f"kg.get_ns('jie').isco{int(row[4])}"))
    kg.add(eval(f"kg.get_ns('jie').isco{row[6]}"), kg.get_ns("rdfs").comment, rdflib.Literal(row[7]))

    # Sub_major
    kg.add(eval(f"kg.get_ns('jie').isco{int(row[4])}"), kg.get_ns("rdf").type, kg.get_ns("jid").ISCO_sub_major)
    kg.add(eval(f"kg.get_ns('jie').isco{int(row[4])}"), kg.get_ns("jip").falls_under, eval(f"kg.get_ns('jie').isco{row[2]}"))
    kg.add(eval(f"kg.get_ns('jie').isco{int(row[4])}"), kg.get_ns("rdfs").comment, rdflib.Literal(row[5]))

    # Major
    kg.add(eval(f"kg.get_ns('jie').isco{row[2]}"), kg.get_ns("rdf").type, kg.get_ns("jid").ISCO_major)
    kg.add(eval(f"kg.get_ns('jie').isco{row[2]}"), kg.get_ns("rdfs").comment, rdflib.Literal(row[3]))
    
# Properties
kg.add(kg.get_ns("jip").offers_position, kg.get_ns("owl").inverseOf, kg.get_ns("jip").position_is_offered_by)

kg.add(kg.get_ns("jip").has_worked_at, kg.get_ns("owl").inverseOf, kg.get_ns("jip").has_employed)

kg.add(kg.get_ns("jip").supersedes, kg.get_ns("owl").inverseOf, kg.get_ns("jip").subsedes)

kg.add(kg.get_ns("jip").falls_under, kg.get_ns("owl").inverseOf, kg.get_ns("jip").encompasses)

# Falls under is transitive
kg.add(kg.get_ns("jip").falls_under, kg.get_ns("rdf").type, kg.get_ns("owl").TransitiveProperty)

In [9]:
measure = kglab.Measure()
measure.measure_graph(kg)

print("edges before inference", measure.get_edge_count())
print("nodes before inference", measure.get_node_count())

# Do all the nifty inference
kg.infer_owlrl_closure()

measure.measure_graph(kg)

print()
print("edges after inference", measure.get_edge_count())
print("nodes after inference", measure.get_node_count())

edges before inference 1879
nodes before inference 641

edges after inference 9514
nodes after inference 1279


In [10]:
with open("../../dataset/final_dataset/anon_cvs.json", 'r', encoding="utf-8") as f:
    data = json.load(f)

In [11]:
def add_triple(kg, s, p, o, literal=False):

    # Non-existent node
    if not literal and not len(o[1]):
        return 1

    # Some nodes start with a digit, which is not allowed
    if not literal and o[1][0].isdigit():
        o = list(o)       
        o[1] = "_" + o[1]

        if len(o[1]) == 2:
            o[1] = "int" + o[1]

    try:
    
        if not literal:      
            # Sometimes cleaning leads to a double underscore
            # which counts as a python special character
            if o[1].startswith("__"):
                o = list(o)
                o[1] = o[1][1:]

            # Sometimes all cleaning leads to a string
            # that only contains underscores
            if set(o[1]) == {"_"}:
                return 1
                
            kg.add(eval(f"kg.get_ns('{s[0]}')." + s[1]), 
                   eval(f"kg.get_ns('{p[0]}')." + p[1]), 
                   eval(f"kg.get_ns('{o[0]}')." + o[1]))
        else:
            # Replace nan with 0
            if np.isnan(o):
                o = 0
                
            kg.add(eval(f"kg.get_ns('{s[0]}')." + s[1]), 
                   eval(f"kg.get_ns('{p[0]}')." + p[1]),
                   rdflib.Literal(int(o), datatype=kg.get_ns("xsd").integer))

    except:
        print(s, p, o)

    return 0

def prepare_node(text):
    if (type(text) == float and np.isnan(text)) or (not text):
        return "none"
    
    text = text.lower()
    
    # "separating" characters become underscores
    cleaned_text = re.sub(r'[ /\\.,:;()&-]+', '_', text)
    
    # All other non-whitespace characters get removed
    final_text = re.sub(r'[^a-zA-Z0-9_]', '', cleaned_text)

    if final_text in ["import", "global", "is", "as", "or", 
                      "and", "in", "for", "while", "lambda",
                      "return", "if", "else", "encode"]:
        final_text = "_" + final_text
    
    return final_text

country_map = {4: "denmark"}

education_map = {8: "Masters_degree"}

fluency_map = {0: "speaks_none", 
               1: "speaks_beginner", 
               2: "speaks_intermediate",
               3: "speaks_advanced",
               4: "speaks_fluent"}

language_map = {"da": "danish", "de": "german", "en": "english", 
                "is": "icelandic", "sv": "swedish", "it": "italian",
                "es": "spanish", "nl": "dutch", "no": "norwegian",
                "fa": "farsi", "ru": "russian", "pl": "polish", 
                "fr": "french", "or": "none"}

for candidate in tqdm(data):
    # Clean and gather data
    name = prepare_node(candidate["headline"])
    country = country_map.get(candidate["countryid"], "unknown")
    education = education_map.get(candidate["educationlevel"], "unknown")

    cv_id = "candidate_" + candidate["cvid"]

    # Add triples
    add_triple(kg, ('jie', cv_id), ("rdf", "type"), ('jie', name))
    add_triple(kg, ('jie', cv_id), ("jip", "has_education_level"), ("jie", education))
    add_triple(kg, ('jie', cv_id), ("jip", "number_of_jobs"), candidate["jobexperiences"], literal=True)
    add_triple(kg, ('jie', cv_id), ("jip", "managerial_experience"), candidate["mgrexperiences"], literal=True)
    add_triple(kg, ('jie', cv_id), ("jip", "lives_in"), ("jie", country))
    
    # Add looped triples
    if type(candidate["job_history"]) != float:
        for job in candidate["job_history"]:
            job = prepare_node(job)
            add_triple(kg, ('jie', cv_id), ('jip', "has_worked_position"), ('jie', job))

    if type(candidate["educations"]) != float:
        for education in candidate["educations"]:
            edu = prepare_node(education["name"])
            if education["type"] == "education":
                add_triple(kg, ('jie', cv_id), ('jip', "has_education"), ('jie', edu))
            elif education["type"] == "course":
                add_triple(kg, ('jie', cv_id), ('jip', "has_course"), ('jie', edu))
            else:
                pass

    if type(candidate["languages"]) != float:
        for language in candidate["languages"]:
            add_triple(kg, ('jie', cv_id), 
                       ('jip', fluency_map.get(language["level"], language["level"])), 
                       ('jie', language_map.get(language["code"], language["code"])))

    if type(candidate["jobtitles"]) != float:
        for title in candidate["jobtitles"]:
            title = prepare_node(title)
            add_triple(kg, ('jie', cv_id), ('jip', "has_job_title"), ('jie', title))

    if type(candidate["keywords"]) != float:
        for keyword in candidate["keywords"]:
            keyword = prepare_node(keyword)
            add_triple(kg, ('jie', cv_id), ('jip', "has_keyword"), ('jie', keyword))

  0%|          | 0/83804 [00:00<?, ?it/s]

In [12]:
measure = kglab.Measure()
measure.measure_graph(kg)

print("edges before inference", measure.get_edge_count())
print("nodes before inference", measure.get_node_count())

# Do all the nifty inference
kg.infer_owlrl_closure()

measure.measure_graph(kg)

print()
print("edges after inference", measure.get_edge_count())
print("nodes after inference", measure.get_node_count())

edges before inference 2205177
nodes before inference 317087

edges after inference 4726406
nodes after inference 317177


In [13]:
for listing in df["triples"]:

    triples = eval(listing)

    for triple in triples:
        i = [re.sub(r'[^A-Za-z0-9]', '_', str(text)).lower() for text in triple]
        i = ["c" + text if text == "global" else text for text in i]

        if len(i) != 3:
            continue
        
        if i[2].isdigit():
            kg.add(eval(f"kg.get_ns('jie').{i[0]}"), 
                   eval(f"kg.get_ns('jip').{i[1]}"), 
                   rdflib.Literal(int(i[2]), datatype=kg.get_ns("xsd").integer))
        elif re.fullmatch(r"\d{4}_\d{2}_\d{2}", i[2]):
            pass
        elif i[2][0].isdigit():
            pass
        elif i[0][0].isdigit():
            pass
        else:
            kg.add(eval(f"kg.get_ns('jie').{i[0]}"), eval(f"kg.get_ns('jip').{i[1]}"), eval(f"kg.get_ns('jie').{i[2]}"))

In [14]:
measure = kglab.Measure()
measure.measure_graph(kg)

print("edges before inference", measure.get_edge_count())
print("nodes before inference", measure.get_node_count())

# Do all the nifty inference
kg.infer_owlrl_closure()

measure.measure_graph(kg)

print()
print("edges after inference", measure.get_edge_count())
print("nodes after inference", measure.get_node_count())

edges before inference 2522709
nodes before inference 317763

edges after inference 5046285
nodes after inference 317959


## Descriptives

In [15]:
def graph_info(G):
    info = [
        f"Name: {G.name}",
        f"Type: {type(G).__name__}",
        f"Nodes: {G.number_of_nodes()}",
        f"Edges: {G.number_of_edges()}",
        f"Is directed: {G.is_directed()}",
    ]
    return "\n".join(info)

In [16]:
# Run your SPARQL query
sparql = """
SELECT ?subject ?object WHERE {
  ?subject ?pred ?object .
}
"""
results = kg.query(sparql)

# Build a NetworkX graph manually
G = nx.DiGraph()

for row in results:
    subject = str(row.subject)
    obj = str(row.object)
    G.add_edge(subject, obj)

print(graph_info(G))

Name: 
Type: DiGraph
Nodes: 317959
Edges: 2402631
Is directed: True


In [17]:
np.mean(list(dict(G.degree()).values()))

np.float64(15.11283530266481)

In [18]:
# Access the underlying rdflib graph
graph = kg.rdf_graph()

# Extract predicates
predicates = [str(p) for (_, p, _) in graph]

# Count occurrences
predicate_counts = Counter(predicates)

len(predicate_counts), predicate_counts

(221,
 Counter({'http://company.com/property/has_keyword': 689789,
          'http://company.com/property/has_job_title': 445778,
          'http://www.w3.org/2002/07/owl#sameAs': 317959,
          'http://company.com/property/has_worked_position': 253649,
          'http://company.com/property/has_education': 146152,
          'http://www.w3.org/1999/02/22-rdf-syntax-ns#type': 85940,
          'http://company.com/property/number_of_jobs': 83804,
          'http://company.com/property/managerial_experience': 83804,
          'http://company.com/property/has_education_level': 83804,
          'http://company.com/property/lives_in': 83804,
          'http://company.com/property/speaks_fluent': 69296,
          'http://company.com/property/speaks_advanced': 57093,
          'http://company.com/property/speaks_intermediate': 43396,
          'http://company.com/property/speaks_beginner': 41444,
          'http://company.com/property/has_course': 24006,
          'http://company.com/propert

In [19]:
# create a Tensor representation (replaces SubgraphTensor)
kgt = kg.rdf_graph()

with open(f"../outputs/inferred_outputs/kg_{model}_{prompt}.edgelist", "w+", encoding="utf-8") as f:
    
    for i, (s, p, o) in tqdm(enumerate(kgt), total=len(kgt)):
            s_label = kg.n3fy(s)
            p_label = kg.n3fy(p)
            o_label = kg.n3fy(o)

            f.write(f"['{s_label}', '{p_label}', '{o_label}']\n")

  0%|          | 0/2523576 [00:00<?, ?it/s]